## ETL of GeoSpatial data from US Census to Hugging Face

The US Census has restrictions on how it serves files.  So we'll copy them elsewhere first.


## Setup


In [9]:
import os
from google.colab import userdata


## Download shapefiles


In [1]:
# Define the remote URLs
roads_url = "https://www2.census.gov/geo/tiger/TIGER2024/ROADS/tl_2024_35006_roads.zip"
states_url = "https://www2.census.gov/geo/tiger/TIGER2024/STATE/tl_2024_us_state.zip"

roads_url, states_url

('https://www2.census.gov/geo/tiger/TIGER2024/ROADS/tl_2024_35006_roads.zip',
 'https://www2.census.gov/geo/tiger/TIGER2024/STATE/tl_2024_us_state.zip')

In [2]:
! curl -L -O {roads_url}
! curl -L -O {states_url}


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2368k  100 2368k    0     0  1979k      0  0:00:01  0:00:01 --:--:-- 1982k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 9721k  100 9721k    0     0  8577k      0  0:00:01  0:00:01 --:--:-- 8579k


In [3]:
!ls -l *.zip


-rw-r--r-- 1 root root 2425481 Nov 18 12:40 tl_2024_35006_roads.zip
-rw-r--r-- 1 root root 9954307 Nov 18 12:40 tl_2024_us_state.zip


In [4]:
! mkdir -p TIGER2024/ROADS/
! mkdir -p TIGER2024/STATE/


In [6]:
! mv ./tl_2024_35006_roads.zip ./TIGER2024/ROADS/
! mv ./tl_2024_us_state.zip ./TIGER2024/STATE/


/bin/bash: line 1: tree: command not found


In [7]:
! find ./TIGER2024/

./TIGER2024/
./TIGER2024/STATE
./TIGER2024/STATE/tl_2024_us_state.zip
./TIGER2024/ROADS
./TIGER2024/ROADS/tl_2024_35006_roads.zip


## Push to Hugging Face


## Configure settings and authenticate to Hugging Face


In [10]:
hf_account = os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account


'rwcitek'

In [11]:
hf_repo = os.environ["HF_REPO"] = "DUC-2026"
hf_repo


'DUC-2026'

In [12]:
hf_file = os.environ["HF_FILE"] = "TIGER2024/"
hf_file


'TIGER2024/'

In [13]:
# Retrieve the token from Colab's secret manager
_ = hf_token = os.environ["HF_TOKEN"] = userdata.get('hf_token')
f"{_[:3]} ... {_[-4:]}"


'hf_ ... dQPb'

In [14]:
# Log in automatically without an interactive prompt
! hf auth login --token ${HF_TOKEN}


Hint: A new version of huggingface_hub (2.4.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Colab` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Push the parquet file to Hugging Face


In [17]:
%%capture hf_upload
%%bash

TARGET_REPO="${HF_ACCOUNT}/${HF_REPO}"
LOCAL_SOURCE="./${HF_FILE}"
REMOTE_DESTINATION="./${HF_FILE}"

# Comment out the echo to make this live
# echo DRY-RUN: \
hf upload \
  --type dataset \
  "${TARGET_REPO}" \
  "${LOCAL_SOURCE}" \
  "${REMOTE_DESTINATION}"


In [18]:
print(hf_upload.stdout)


✓ Uploaded
  url: https://huggingface.co/datasets/rwcitek/DUC-2026/commit/8997a1c39566f9269615644d925206c57f8328e3



In [28]:
%%capture hf_repo_list
%%bash

TARGET_REPO="${HF_ACCOUNT}/${HF_REPO}"
LOCAL_SOURCE="./${HF_FILE}"
REMOTE_DESTINATION="./${HF_FILE}"

# Comment out the echo to make this live
echo DRY-RUN: \
hf datasets  \
  list -R \
  "${TARGET_REPO}" \


In [29]:
print(hf_repo_list.stdout)


              2026-11-18 12:54:41  TIGER2024/
              2026-11-18 12:54:41  TIGER2024/ROADS/
              2026-11-18 12:54:41  TIGER2024/STATE/
        2504  2026-11-17 05:07:06  .gitattributes
          28  2026-11-17 05:07:06  README.md
     2425481  2026-11-18 12:54:41  TIGER2024/ROADS/tl_2024_35006_roads.zip
     9954307  2026-11-18 12:54:41  TIGER2024/STATE/tl_2024_us_state.zip
     7464556  2026-11-17 05:09:18  sub-est2025.csv

